# GPT-2 XL: Accessibility Domain Knowledge

48 layers, d_model=1600, d_mlp=6400, 25 heads.
The big one. Does GPT-2 XL finally *know* accessibility concepts
that Medium couldn't retrieve?

In [ ]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [2]:
model, info = load_model("gpt2-xl")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-xl into HookedTransformer
Loaded gpt2-xl on mps
  48 layers | 25 heads | d_model=1600 | d_mlp=6400 | sequential attn+MLP


In [3]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: gpt2-xl
Prompt: "The W in WCAG stands for"
Target: " Web" (token 5313)
Final prediction: " Web"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         example             3957           0.000047    
1         example             5007           0.000044    
2         example             6799           0.000034    
3         example             5620           0.000040    
4         example             3620           0.000058    
5         a                   4225           0.000049    
6         example             4975           0.000041    
7         example             7552           0.000025    
8         example             8526           0.000021    
9         example             18539          0.000007    
10        Burg                20065          0.000005    
11        something           16088          0.000008    
12        '                   15315      

In [ ]:
for label, data in results.items():
    print(f"\n--- {label} ---")
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [ ]:
model_name = "gpt2-xl"

for label, data in results.items():
    data["lens"].to_csv(f"../../results/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/{model_name}")

In [6]:
unload(model)

Model unloaded, memory cleared
